# M02b · Variables aléatoires et lois usuelles — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookMaths/blob/main/chapitres/m02b_variables_aleatoires/03_notebook.ipynb)

La partie 0 vérifie tes exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wbm.check`). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/m02b_variables_aleatoires/03_notebook.ipynb`, créée par `python tools/start_chapter.py M02b`) ; la version du dépôt est mise à jour par Claude. Sur Colab, le badge ouvre la version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | M02b.1 à M02b.10, M02b.14 | Les exercices papier (vérification) | ✏️ 🔗 | | |
| A | M02b.18 | Lois binomiale et de Poisson · `binomial_pmf`, `poisson_pmf` | 🔨 | ★ | 20 |
|  | M02b.19 | Densité et fonction de répartition de la loi normale (avec `math.erf`) · `normal_pdf`, `normal_cdf` | 🔨 | ★ | 20 |
|  | M02b.20 | Simuler par inversion de la fonction de répartition · `inverse_transform_sample` | 🔨 | ★★ | 20 |
|  | M02b.21 | `scipy.stats` : pmf, pdf, cdf, ppf et rvs | 📦 | ★ | 15 |
| B | M02b.22 | Prédire : la variance de la moyenne de quatre dés | 🔮 | ★ | 10 |
|  | M02b.23 | Binomiale et Poisson côte à côte quand n grandit | 🔬 | ★★ | 20 |
|  | M02b.24 | Une loi jointe en carte de chaleur, ses marginales sur les bords | 🎨 | ★ | 20 |
| C | M02b.25 | Chasse à l'erreur : une densité prise pour une probabilité, σ et σ² | 🐛 | ★ | 15 |
|  | M02b.26 | Documenter une loi : docstring et exemples testés (`doctest`) | 🛠️ | ★ | 15 |
|  | M02b.27 | Défi : tirer des valeurs normales avec seulement des tirages uniformes | 🏆 | ★★★ | 30 |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (a few seconds per cell); False = bigger experiments
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookMaths.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookMaths")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WBM_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wbm").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wbm").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier de workbookMaths.")

os.environ["WBM_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wbm  # noqa: E402

cfg = wbm.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M02b")  # your own library

## Objectifs et rappel express

- Vérifier tes calculs papier (partie 0) : lois binomiale, géométrique, de Poisson, uniforme, exponentielle et normale, espérances et variances, loi jointe et indépendance, variance d'une somme, intégrale double, changement de variable, erreurs d'annotation.
- Écrire les cinq fonctions de `mymath/distributions.py` et les tester, puis utiliser `scipy.stats`.
- Prédire la variance d'une moyenne, mesurer l'écart entre binomiale et Poisson, dessiner une loi jointe, chasser des confusions, documenter une loi, tirer des normales avec des uniformes.

**Rappel express.** $\mathcal{B}(n, p)$ : $\binom{n}{k}p^k(1-p)^{n-k}$, $\mathbb{E} = np$, $\mathbb{V} = np(1-p)$ ; $\mathcal{P}(\lambda)$ : $e^{-\lambda}\frac{\lambda^k}{k!}$, $\mathbb{E} = \mathbb{V} = \lambda$ ; $\mathcal{E}(\lambda)$ : $P(T > t) = e^{-\lambda t}$, $\mathbb{E} = \frac{1}{\lambda}$ ; $\mathcal{N}(\mu, \sigma^2)$ : $P(X \le x) = \Phi\big(\frac{x - \mu}{\sigma}\big)$. $\mathbb{V}(X + Y) = \mathbb{V}(X) + \mathbb{V}(Y)$ si $X$ et $Y$ sont indépendantes. Inversion : $F^{-1}(U)$ suit la loi de $F$.

## Partie 0 · Vérifier tes exercices papier (✏️ M02b.1 à M02b.10, 🔗 M02b.14)

Fais d'abord les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse ici : **la valeur** que tu as trouvée, sous la forme indiquée en commentaire, jamais le calcul qui y mène :
- un entier, ou un nombre avec un point décimal, `-1.25`, arrondi comme le demande la question ;
- une fraction irréductible entre guillemets, `"5/9"` ;
- une expression entre guillemets, dans la variable indiquée ($p$, $x$ ou $y$), avec `**` pour les puissances, `sqrt` et des parenthèses autour d'un dénominateur, `"3/(2*p)"`, `"sqrt(y)/5"` ;
- une liste `[0.25, 0.75]`, un intervalle `"[-2 ; 7]"`, ou `True` / `False`.

Pour la loi normale, utilise la table de $\Phi$ de la fiche (§3) : les réponses attendues sont arrondies de sorte que la table et un calcul exact donnent la même valeur. Les réponses pas encore remplies affichent ⏳.

In [ ]:
from fractions import Fraction   # Fraction(5, 9) is accepted too, as "5/9"
import sympy as sp
p, x, y = sp.symbols("p x y")      # an expression typed without quotes works too

**Ex M02b.1 — Loi de Bernoulli et loi binomiale**

In [ ]:
# Ex M02b.1: the values you found on paper
answer_M02b_1a = ...  # a) a number (4 decimals)
answer_M02b_1b = ...  # b) a number (4 decimals)
answer_M02b_1c = ...  # c) a number (4 decimals)
answer_M02b_1d = ...  # d) an integer
answer_M02b_1e = ...  # e) a number (1 decimal)
answer_M02b_1f = ...  # f) a number (2 decimals)
answer_M02b_1g = ...  # g) a number (4 decimals)
answer_M02b_1h = ...  # h) an integer

for letter, answer in zip("abcdefgh", [answer_M02b_1a, answer_M02b_1b, answer_M02b_1c, answer_M02b_1d, answer_M02b_1e, answer_M02b_1f, answer_M02b_1g, answer_M02b_1h]):
    wbm.check(f"M02b.1{letter}", answer)

**Ex M02b.2 — Loi géométrique : probabilités, et l'espérance par une série**

In [ ]:
# Ex M02b.2: the values you found on paper
answer_M02b_2a = ...  # a) a number (3 decimals)
answer_M02b_2b = ...  # b) a number (3 decimals)
answer_M02b_2c = ...  # c) a number (2 decimals)
answer_M02b_2d = ...  # d) a number (2 decimals)
answer_M02b_2e = ...  # e) a number (1 decimal)
answer_M02b_2f = ...  # f) a number (1 decimal)
answer_M02b_2g = ...  # g) an integer
answer_M02b_2h = ...  # h) an expression in p, as text

for letter, answer in zip("abcdefgh", [answer_M02b_2a, answer_M02b_2b, answer_M02b_2c, answer_M02b_2d, answer_M02b_2e, answer_M02b_2f, answer_M02b_2g, answer_M02b_2h]):
    wbm.check(f"M02b.2{letter}", answer)

**Ex M02b.3 — Loi de Poisson : probabilités, espérance, et la somme de deux variables de Poisson indépendantes**

In [ ]:
# Ex M02b.3: the values you found on paper
answer_M02b_3a = ...  # a) a number (4 decimals)
answer_M02b_3b = ...  # b) a number (4 decimals)
answer_M02b_3c = ...  # c) a number (4 decimals)
answer_M02b_3d = ...  # d) a number (1 decimal)
answer_M02b_3e = ...  # e) a number (4 decimals)
answer_M02b_3f = ...  # f) a number (4 decimals)
answer_M02b_3g = ...  # g) a number (4 decimals)
answer_M02b_3h = ...  # h) a number (1 decimal)

for letter, answer in zip("abcdefgh", [answer_M02b_3a, answer_M02b_3b, answer_M02b_3c, answer_M02b_3d, answer_M02b_3e, answer_M02b_3f, answer_M02b_3g, answer_M02b_3h]):
    wbm.check(f"M02b.3{letter}", answer)

**Ex M02b.4 — Lois uniforme et exponentielle : probabilités et espérance**

In [ ]:
# Ex M02b.4: the values you found on paper
answer_M02b_4a = ...  # a) an irreducible fraction, as text
answer_M02b_4b = ...  # b) an irreducible fraction, as text
answer_M02b_4c = ...  # c) an irreducible fraction, as text
answer_M02b_4d = ...  # d) a number (1 decimal)
answer_M02b_4e = ...  # e) a number (3 decimals)
answer_M02b_4f = ...  # f) a number (4 decimals)
answer_M02b_4g = ...  # g) a number (1 decimal)
answer_M02b_4h = ...  # h) a number (4 decimals)

for letter, answer in zip("abcdefgh", [answer_M02b_4a, answer_M02b_4b, answer_M02b_4c, answer_M02b_4d, answer_M02b_4e, answer_M02b_4f, answer_M02b_4g, answer_M02b_4h]):
    wbm.check(f"M02b.4{letter}", answer)

**Ex M02b.5 — Loi normale : standardiser et lire la fonction Φ**

In [ ]:
# Ex M02b.5: the values you found on paper
answer_M02b_5a = ...  # a) a number (1 decimal)
answer_M02b_5b = ...  # b) a number (4 decimals)
answer_M02b_5c = ...  # c) a number (4 decimals)
answer_M02b_5d = ...  # d) a number (4 decimals)
answer_M02b_5e = ...  # e) a number (4 decimals)
answer_M02b_5f = ...  # f) a number (2 decimals)
answer_M02b_5g = ...  # g) a number (3 decimals)
answer_M02b_5h = ...  # h) a number (4 decimals)

for letter, answer in zip("abcdefgh", [answer_M02b_5a, answer_M02b_5b, answer_M02b_5c, answer_M02b_5d, answer_M02b_5e, answer_M02b_5f, answer_M02b_5g, answer_M02b_5h]):
    wbm.check(f"M02b.5{letter}", answer)

**Ex M02b.6 — Espérance et variance d'une variable continue (théorème de transfert)**

In [ ]:
# Ex M02b.6: the values you found on paper
answer_M02b_6a = ...  # a) an irreducible fraction, as text
answer_M02b_6b = ...  # b) an irreducible fraction, as text
answer_M02b_6c = ...  # c) an irreducible fraction, as text
answer_M02b_6d = ...  # d) an irreducible fraction, as text
answer_M02b_6e = ...  # e) an irreducible fraction, as text
answer_M02b_6f = ...  # f) an irreducible fraction, as text
answer_M02b_6g = ...  # g) a number (4 decimals)
answer_M02b_6h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_M02b_6a, answer_M02b_6b, answer_M02b_6c, answer_M02b_6d, answer_M02b_6e, answer_M02b_6f, answer_M02b_6g, answer_M02b_6h]):
    wbm.check(f"M02b.6{letter}", answer)

**Ex M02b.7 — Un couple de variables discrètes : loi jointe, marginales, indépendance**

In [ ]:
# Ex M02b.7: the values you found on paper
answer_M02b_7a = ...  # a) a number (2 decimals)
answer_M02b_7b = ...  # b) a list (2 decimals)
answer_M02b_7c = ...  # c) a number (2 decimals)
answer_M02b_7d = ...  # d) a number (2 decimals)
answer_M02b_7e = ...  # e) a number (2 decimals)
answer_M02b_7f = ...  # f) True or False
answer_M02b_7g = ...  # g) a number (2 decimals)
answer_M02b_7h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_M02b_7a, answer_M02b_7b, answer_M02b_7c, answer_M02b_7d, answer_M02b_7e, answer_M02b_7f, answer_M02b_7g, answer_M02b_7h]):
    wbm.check(f"M02b.7{letter}", answer)

**Ex M02b.8 — Linéarité et variance d'une somme : sommes et moyennes de variables**

In [ ]:
# Ex M02b.8: the values you found on paper
answer_M02b_8a = ...  # a) an integer
answer_M02b_8b = ...  # b) an integer
answer_M02b_8c = ...  # c) an integer
answer_M02b_8d = ...  # d) an integer
answer_M02b_8e = ...  # e) a number (3 decimals)
answer_M02b_8f = ...  # f) a number (3 decimals)
answer_M02b_8g = ...  # g) a number (4 decimals)
answer_M02b_8h = ...  # h) an integer

for letter, answer in zip("abcdefgh", [answer_M02b_8a, answer_M02b_8b, answer_M02b_8c, answer_M02b_8d, answer_M02b_8e, answer_M02b_8f, answer_M02b_8g, answer_M02b_8h]):
    wbm.check(f"M02b.8{letter}", answer)

**Ex M02b.9 — Une intégrale double sur un rectangle, et une densité jointe**

In [ ]:
# Ex M02b.9: the values you found on paper
answer_M02b_9a = ...  # a) an irreducible fraction, as text
answer_M02b_9b = ...  # b) a number (4 decimals)
answer_M02b_9c = ...  # c) an irreducible fraction, as text
answer_M02b_9d = ...  # d) an irreducible fraction, as text
answer_M02b_9e = ...  # e) an expression in x, as text
answer_M02b_9f = ...  # f) an expression in y, as text
answer_M02b_9g = ...  # g) True or False
answer_M02b_9h = ...  # h) an irreducible fraction, as text

for letter, answer in zip("abcdefgh", [answer_M02b_9a, answer_M02b_9b, answer_M02b_9c, answer_M02b_9d, answer_M02b_9e, answer_M02b_9f, answer_M02b_9g, answer_M02b_9h]):
    wbm.check(f"M02b.9{letter}", answer)

**Ex M02b.10 — Changement de variable : la loi de aX + b, et celle de X²**

In [ ]:
# Ex M02b.10: the values you found on paper
answer_M02b_10a = ...  # a) an interval, as text
answer_M02b_10b = ...  # b) an irreducible fraction, as text
answer_M02b_10c = ...  # c) an irreducible fraction, as text
answer_M02b_10d = ...  # d) a number (4 decimals)
answer_M02b_10e = ...  # e) an irreducible fraction, as text
answer_M02b_10f = ...  # f) an expression in y, as text
answer_M02b_10g = ...  # g) an expression in y, as text
answer_M02b_10h = ...  # h) an integer

for letter, answer in zip("abcdefgh", [answer_M02b_10a, answer_M02b_10b, answer_M02b_10c, answer_M02b_10d, answer_M02b_10e, answer_M02b_10f, answer_M02b_10g, answer_M02b_10h]):
    wbm.check(f"M02b.10{letter}", answer)

**Ex M02b.14 — Combien d'erreurs d'annotation dans un dataset ? Binomiale et Poisson**

In [ ]:
# Ex M02b.14: the values you found on paper
answer_M02b_14a = ...  # a) an integer
answer_M02b_14b = ...  # b) a number (3 decimals)
answer_M02b_14c = ...  # c) a number (4 decimals)
answer_M02b_14d = ...  # d) a number (4 decimals)
answer_M02b_14e = ...  # e) a number (4 decimals)
answer_M02b_14f = ...  # f) a number (4 decimals)
answer_M02b_14g = ...  # g) an integer

for letter, answer in zip("abcdefg", [answer_M02b_14a, answer_M02b_14b, answer_M02b_14c, answer_M02b_14d, answer_M02b_14e, answer_M02b_14f, answer_M02b_14g]):
    wbm.check(f"M02b.14{letter}", answer)

## Partie A · Ta librairie : lois binomiale, de Poisson et normale, simulation par inversion

Fiche §2, §3 et §8. Les cinq fonctions de `mymath/distributions.py`, puis les mêmes lois avec `scipy.stats`.

In [ ]:
# Tools of the notebook (parts A to C)
import math

import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

from wbm.nbtools import checked, drawn, run_pytest_group, verdict


def raised(function, *args):
    """The name of the exception that function(*args) raises, or "aucune" if it returns.

    NotImplementedError (a TODO not done yet) goes through, for wbm.attempt to show ⏳."""
    try:
        function(*args)
    except NotImplementedError:
        raise
    except Exception as exc:  # noqa: BLE001 - any other error is the answer
        return type(exc).__name__
    return "aucune"

### Ex M02b.18 — Lois binomiale et de Poisson · `binomial_pmf`, `poisson_pmf` 🔨 ★ ⏱️ 20 min
**Objectif :** programmer les probabilités des lois binomiale et de Poisson, avec leurs cas limites et leurs cas d'erreur.  
**Prérequis :** fiche §2 ; R8 (coefficients binomiaux) · **mymath :** `distributions.py` · **Parcours :** C, E

> **Mode d'emploi `mymath`** (comme en M02a). Ouvre ta copie `mon_travail/mymath/distributions.py`, lis la docstring de la fonction en entier, remplace la ligne `raise NotImplementedError(...)` par ton code, **enregistre** le fichier, puis exécute les cellules ci-dessous : la première recharge ta librairie. `math.comb`, `math.factorial`, `math.exp`, `math.erf` et NumPy sont permis (importe-les en haut du fichier). Mais pas `scipy.stats` : les tests comparent tes résultats à lui, et un test vérifie que tu ne t'en sers pas. Chaque probabilité ou densité est un `float` de Python (`float(...)` convertit un nombre de NumPy).

Écris `binomial_pmf(k, n, p)` puis `poisson_pmf(k, lam)` (fiche §2). Une valeur de $k$ impossible (négative, ou plus grande que $n$ pour la binomiale) a la probabilité `0.0` : ce n'est pas une erreur. Des paramètres impossibles ($n < 0$, $p$ hors de $[0 ; 1]$, $\lambda < 0$) lèvent une `ValueError`.

La vérification calcule :
- a) `binomial_pmf(3, 10, 0.3)` (6 décimales) ;
- b) la somme des `binomial_pmf(k, 12, 0.35)` pour $k$ de 0 à 12 (6 décimales) ;
- c) `binomial_pmf(11, 10, 0.3)` ;
- d) le nom de l'erreur levée par `binomial_pmf(2, 5, 1.5)` (avec `raised`, de la cellule des outils) ;
- e) `poisson_pmf(4, 2.5)` (6 décimales) ;
- f) `poisson_pmf(0, 0)` ;
- g) le type de `poisson_pmf(1, 3)`, en texte (`"float"` pour un flottant) ;
- h) la somme des `k * poisson_pmf(k, 6.5)` pour $k$ de 0 à 100 (4 décimales).

Puis, dans ta copie : quelle propriété de la loi binomiale b) vérifie-t-il ? Pourquoi h) redonne-t-il le paramètre, et pourquoi s'arrêter à $k = 100$ suffit-il ?

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M02b")  # reload your saved file

In [ ]:
with checked("M02b.18a"):
    wbm.check("M02b.18a", mymath.distributions.binomial_pmf(3, 10, 0.3), computed=True)
with checked("M02b.18b"):
    wbm.check("M02b.18b", sum(mymath.distributions.binomial_pmf(k, 12, 0.35) for k in range(13)), computed=True)
with checked("M02b.18c"):
    wbm.check("M02b.18c", mymath.distributions.binomial_pmf(11, 10, 0.3), computed=True)
with checked("M02b.18d"):
    wbm.check("M02b.18d", raised(mymath.distributions.binomial_pmf, 2, 5, 1.5), computed=True)
with checked("M02b.18e"):
    wbm.check("M02b.18e", mymath.distributions.poisson_pmf(4, 2.5), computed=True)
with checked("M02b.18f"):
    wbm.check("M02b.18f", mymath.distributions.poisson_pmf(0, 0), computed=True)
with checked("M02b.18g"):
    wbm.check("M02b.18g", type(mymath.distributions.poisson_pmf(1, 3)).__name__, computed=True)
with checked("M02b.18h"):
    wbm.check("M02b.18h", sum(k * mymath.distributions.poisson_pmf(k, 6.5) for k in range(101)), computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m02b_distributions.py", "test_binomial_pmf_ or test_poisson_pmf_ or test_distributions_uses")

### Ex M02b.19 — Densité et fonction de répartition de la loi normale (avec `math.erf`) · `normal_pdf`, `normal_cdf` 🔨 ★ ⏱️ 20 min
**Objectif :** programmer la densité et la fonction de répartition de la loi normale, et vérifier que l'une est la dérivée de l'autre.  
**Prérequis :** fiche §3 ; M02a.19 (erf) · **mymath :** `distributions.py` · **Parcours :** C, E

Écris `normal_pdf(x, mu=0.0, sigma=1.0)` et `normal_cdf(x, mu=0.0, sigma=1.0)` (fiche §3). `sigma` est l'**écart-type**. Pour $\Phi$, la fonction d'erreur `math.erf` fait le travail : $\Phi(z) = \frac{1}{2}\left(1 + \operatorname{erf}\frac{z}{\sqrt{2}}\right)$, après avoir standardisé. Les deux fonctions lèvent une `ValueError` si $\sigma \le 0$.

La vérification calcule :
- a) `normal_pdf(1)` (6 décimales) ;
- b) `normal_pdf(3, 3, 0.2)` (4 décimales) ;
- c) `normal_cdf(1.5)` (4 décimales), à comparer avec la table de la fiche ;
- d) `normal_cdf(4, 7, 2)` (4 décimales) ;
- e) `normal_cdf(2.5) - normal_cdf(0.5)` (4 décimales) ;
- f) le nom de l'erreur levée par `normal_pdf(0, 0, 0)` ;
- g) le type de `normal_cdf(0.3)`, en texte ;
- h) le taux d'accroissement `(normal_cdf(0.7 + 1e-5) - normal_cdf(0.7 - 1e-5)) / 2e-5` (5 décimales).

Dans ta copie : b) dépasse 1. Est-ce une erreur ? Compare h) avec `normal_pdf(0.7)` : quel théorème de M02a cela illustre-t-il ?

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M02b")  # reload your saved file

In [ ]:
with checked("M02b.19a"):
    wbm.check("M02b.19a", mymath.distributions.normal_pdf(1), computed=True)
with checked("M02b.19b"):
    wbm.check("M02b.19b", mymath.distributions.normal_pdf(3, 3, 0.2), computed=True)
with checked("M02b.19c"):
    wbm.check("M02b.19c", mymath.distributions.normal_cdf(1.5), computed=True)
with checked("M02b.19d"):
    wbm.check("M02b.19d", mymath.distributions.normal_cdf(4, 7, 2), computed=True)
with checked("M02b.19e"):
    wbm.check("M02b.19e", mymath.distributions.normal_cdf(2.5) - mymath.distributions.normal_cdf(0.5), computed=True)
with checked("M02b.19f"):
    wbm.check("M02b.19f", raised(mymath.distributions.normal_pdf, 0, 0, 0), computed=True)
with checked("M02b.19g"):
    wbm.check("M02b.19g", type(mymath.distributions.normal_cdf(0.3)).__name__, computed=True)
with checked("M02b.19h"):
    slope = (mymath.distributions.normal_cdf(0.7 + 1e-5) - mymath.distributions.normal_cdf(0.7 - 1e-5)) / 2e-5
    wbm.check("M02b.19h", slope, computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m02b_distributions.py", "test_normal_pdf_ or test_normal_cdf_")

### Ex M02b.20 — Simuler par inversion de la fonction de répartition · `inverse_transform_sample` 🔨 ★★ ⏱️ 20 min
**Objectif :** programmer la simulation par inversion, et s'en servir pour une loi continue et pour une loi discrète.  
**Prérequis :** fiche §8 ; M02a §7 (quantiles) · **mymath :** `distributions.py` · **Parcours :** C

Écris `inverse_transform_sample(inverse_cdf, n, seed=0)` (fiche §8) : un générateur `np.random.default_rng(seed)`, puis $n$ nombres uniformes **dans l'ordre** (`rng.random(n)`), puis `inverse_cdf` appliquée à **chacun** (elle prend un seul nombre). Le résultat est un array NumPy de flottants, de forme `(n,)` ; $n < 1$ lève une `ValueError`.

La loi géométrique de paramètre $p$ se tire aussi par inversion, avec la fonction $u \mapsto \left\lceil \frac{\ln(1 - u)}{\ln(1 - p)} \right\rceil$ (`math.ceil` arrondit à l'entier supérieur).

La vérification calcule :
- a) le premier élément de `inverse_transform_sample(lambda u: 10 * u, 4)` (6 décimales) ;
- b) la moyenne de 100 000 tirages de la loi exponentielle de paramètre 3, obtenus avec $u \mapsto -\frac{\ln(1 - u)}{3}$ (graine 0 ; 4 décimales) ;
- c) le nombre de ces 100 000 tirages qui dépassent 1 (un entier) ;
- d) le nom de l'erreur levée par `inverse_transform_sample(lambda u: u, 0)` ;
- e) la moyenne de 50 000 tirages de la loi géométrique de paramètre 0,2 (graine 0 ; 2 décimales) ;
- f) la proportion de ces 50 000 tirages égaux à 1 (3 décimales) ;
- g) le nom du type des éléments de `inverse_transform_sample(lambda u: int(6 * u) + 1, 3)`, en texte (`.dtype.name`) ;
- h) le premier élément de `inverse_transform_sample(lambda u: 10 * u, 5, seed=3)` (6 décimales).

Dans ta copie : compare b), c) et f) à leurs valeurs théoriques ($\frac{1}{3}$, $100\,000\,e^{-3}$, 0,2). Démontre que, pour $U$ uniforme, $P\left(\left\lceil \frac{\ln(1 - U)}{\ln(1 - p)} \right\rceil > k\right) = (1 - p)^k$ (indication : $\lceil t \rceil > k$ équivaut à $t > k$ pour un entier $k$, et $\ln(1 - p) < 0$).

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="M02b")  # reload your saved file

In [ ]:
with checked("M02b.20a"):
    wbm.check("M02b.20a", mymath.distributions.inverse_transform_sample(lambda u: 10 * u, 4)[0], computed=True)
with checked("M02b.20b"):
    waits = mymath.distributions.inverse_transform_sample(lambda u: -math.log(1 - u) / 3, 100_000)
    wbm.check("M02b.20b", waits.mean(), computed=True)
with checked("M02b.20c"):
    waits = mymath.distributions.inverse_transform_sample(lambda u: -math.log(1 - u) / 3, 100_000)
    wbm.check("M02b.20c", int(np.sum(waits > 1)), computed=True)
with checked("M02b.20d"):
    wbm.check("M02b.20d", raised(mymath.distributions.inverse_transform_sample, lambda u: u, 0), computed=True)
with checked("M02b.20e"):
    ranks = mymath.distributions.inverse_transform_sample(
        lambda u: math.ceil(math.log(1 - u) / math.log(1 - 0.2)), 50_000)
    wbm.check("M02b.20e", ranks.mean(), computed=True)
with checked("M02b.20f"):
    ranks = mymath.distributions.inverse_transform_sample(
        lambda u: math.ceil(math.log(1 - u) / math.log(1 - 0.2)), 50_000)
    wbm.check("M02b.20f", np.mean(ranks == 1), computed=True)
with checked("M02b.20g"):
    wbm.check("M02b.20g", mymath.distributions.inverse_transform_sample(lambda u: int(6 * u) + 1, 3).dtype.name,
              computed=True)
with checked("M02b.20h"):
    wbm.check("M02b.20h", mymath.distributions.inverse_transform_sample(lambda u: 10 * u, 5, seed=3)[0], computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_m02b_distributions.py", "test_inverse_transform_sample_")

**La figure** : l'histogramme des tirages par inversion et la densité de la loi exponentielle.

In [ ]:
with checked("M02b.20 figure"):
    draws = mymath.distributions.inverse_transform_sample(lambda u: -math.log(1 - u) / 3, 20_000)
    grid_20 = np.linspace(0, 2, 200)
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.hist(draws, bins=np.linspace(0, 2, 51), density=True, alpha=0.5, label="20 000 tirages par inversion")
    ax.plot(grid_20, 3 * np.exp(-3 * grid_20), color="black", label="densité 3e^(−3t)")
    ax.set(xlabel="t", title="La loi exponentielle de paramètre 3, simulée")
    ax.legend()
    plt.show()

### Ex M02b.21 — `scipy.stats` : pmf, pdf, cdf, ppf et rvs 📦 ★ ⏱️ 15 min
**Objectif :** utiliser les lois de `scipy.stats` et leurs méthodes, en évitant les pièges de leurs paramètres `loc` et `scale`.  
**Prérequis :** fiche §2, §3 et §8 · **Parcours :** C, E

Chaque loi de `scipy.stats` est un objet qu'on « fige » avec ses paramètres, puis qu'on interroge : `law = stats.binom(10, 0.3)`, puis `law.pmf(k)` (loi discrète), `law.pdf(x)` (densité), `law.cdf(x)` (fonction de répartition), `law.ppf(q)` (la fonction quantile, $F^{-1}$), `law.mean()`, `law.var()`, et `law.rvs(size=..., random_state=...)` pour tirer des valeurs. Les lois continues ont deux paramètres communs, `loc` (un décalage) et `scale` (un facteur d'échelle) : lis `help(stats.expon)` et `help(stats.uniform)` avant e) et h).

Calcule :
- a) $P(X = 3)$ pour $X \sim \mathcal{B}(20 ; 0{,}15)$ (4 décimales) ;
- b) $P(X \le 2)$ pour $X \sim \mathcal{P}(4{,}5)$ (4 décimales) ;
- c) $P(X \le 130)$ pour $X \sim \mathcal{N}(100, 15^2)$ (4 décimales) ;
- d) le percentile 90 de $\mathcal{N}(100, 15^2)$ (2 décimales) ;
- e) l'espérance de la loi exponentielle de **paramètre** $\lambda = 2$, avec `stats.expon` (1 décimale) ;
- f) le nombre $z$ tel que $\Phi(z) = 0{,}975$ (4 décimales) ;
- g) la moyenne de 10 000 tirages de $\mathcal{P}(4{,}5)$, avec `random_state=0` (4 décimales) ;
- h) la variance de la loi uniforme sur $[2 ; 8]$, avec `stats.uniform` (1 décimale).

Dans ta copie : quelle formule de la fiche donne e) et h) ? Que vaudrait d) avec la table de $\Phi$ (indication : $\Phi(1{,}2816) \approx 0{,}9$) ?

In [ ]:
answer_M02b_21a = ...  # a) stats.binom(...).pmf(...)
answer_M02b_21b = ...  # b) stats.poisson(...).cdf(...)
answer_M02b_21c = ...  # c) stats.norm(...).cdf(...)
answer_M02b_21d = ...  # d) stats.norm(...).ppf(...)
answer_M02b_21e = ...  # e) stats.expon(...).mean()
answer_M02b_21f = ...  # f) stats.norm().ppf(...)
answer_M02b_21g = ...  # g) the mean of stats.poisson(...).rvs(size=10_000, random_state=0)
answer_M02b_21h = ...  # h) stats.uniform(...).var()

In [ ]:
for letter, answer in zip("abcdefgh", [answer_M02b_21a, answer_M02b_21b, answer_M02b_21c, answer_M02b_21d,
                                        answer_M02b_21e, answer_M02b_21f, answer_M02b_21g, answer_M02b_21h]):
    wbm.check(f"M02b.21{letter}", answer)

## Partie B · Une prédiction, une expérience et une figure

Fiche §2, §5 et §6. La variance d'une moyenne, la vitesse de l'approximation de Poisson, une loi jointe en carte de chaleur.

### Ex M02b.22 — Prédire : la variance de la moyenne de quatre dés 🔮 ★ ⏱️ 10 min
**Objectif :** prévoir l'effet d'une somme et d'une moyenne de variables indépendantes sur la variance et l'écart-type, puis le vérifier par simulation.  
**Prérequis :** fiche §6 · **Parcours :** C, E

On lance quatre dés équilibrés à 8 faces (numérotées de 1 à 8), indépendants. Pour chaque question, choisis : (1) la même ; (2) multipliée (ou divisée) par 2 ; (3) multipliée (ou divisée) par 4 ; (4) multipliée (ou divisée) par 16.
- a) la variance de la **moyenne** des quatre dés, comparée à la variance d'un dé ;
- b) l'écart-type de cette moyenne, comparé à l'écart-type d'un dé ;
- c) la variance de la **somme** des quatre dés, comparée à la variance d'un dé ;
- d) la variance de **4 fois** le résultat d'un seul dé, comparée à la variance d'un dé.

**Avant d'exécuter quoi que ce soit**, écris tes quatre prédictions (le **numéro du choix**, de 1 à 4), vérifie-les, puis lance l'expérience : 100 000 lancers de quatre dés.

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
prediction_M02b_22a = ...  # a) 1, 2, 3 or 4, written BEFORE running the experiment below
prediction_M02b_22b = ...  # b) 1, 2, 3 or 4
prediction_M02b_22c = ...  # c) 1, 2, 3 or 4
prediction_M02b_22d = ...  # d) 1, 2, 3 or 4

In [ ]:
for letter, prediction in zip("abcd", [prediction_M02b_22a, prediction_M02b_22b, prediction_M02b_22c,
                                      prediction_M02b_22d]):
    wbm.check(f"M02b.22{letter}", prediction)

**L'expérience.** Elle ne s'exécute qu'une fois tes quatre prédictions écrites.

In [ ]:
if any(answer is ... or answer is None for answer in [prediction_M02b_22a, prediction_M02b_22b, prediction_M02b_22c, prediction_M02b_22d]):
    print("⏳ Écris d'abord tes quatre prédictions, puis exécute à nouveau.")
else:
    rng_22 = np.random.default_rng(0)
    dice = rng_22.integers(1, 9, size=(100_000, 4))          # 100 000 rows of 4 eight-sided dice
    one = dice[:, 0]
    print(f"variance d'un dé                        : {one.var():.3f}")
    print(f"variance de la moyenne des 4 dés         : {dice.mean(axis=1).var():.3f}")
    print(f"écart-type d'un dé / celui de la moyenne : {one.std() / dice.mean(axis=1).std():.2f}")
    print(f"variance de la somme des 4 dés           : {dice.sum(axis=1).var():.3f}")
    print(f"variance de 4 fois un seul dé            : {(4 * one).var():.3f}")

**Après l'expérience**, dans ta copie : écris les quatre résultats avec les formules de la fiche §6 ($\mathbb{V}(X_1 + \cdots + X_n)$, $\mathbb{V}(aX)$, $\mathbb{V}(\bar{X})$). Pourquoi c) et d) diffèrent-ils, alors que la somme de quatre dés et 4 fois un dé ont la même espérance ?

### Ex M02b.23 — Binomiale et Poisson côte à côte quand n grandit 🔬 ★★ ⏱️ 20 min
**Objectif :** mesurer l'écart entre la loi binomiale $\mathcal{B}(n, \frac{\lambda}{n})$ et la loi de Poisson, et trouver à quelle vitesse il diminue.  
**Prérequis :** fiche §2 ; Ex M02b.13 (la limite) · **Parcours :** C

Pour chaque $n$ de `N_LIST`, on compare la loi binomiale $\mathcal{B}\big(n, \frac{3}{n}\big)$ et la loi de Poisson $\mathcal{P}(3)$ (cellule fournie : `N_LIST`, `LAM` $= 3$). L'**écart** entre les deux est le plus grand des écarts entre leurs probabilités, $\max_{0 \le k \le n} \big\lvert P(B = k) - P(N = k) \big\rvert$. Calcule-le avec `scipy.stats` (`stats.binom(n, p).pmf(ks)` et `stats.poisson(LAM).pmf(ks)` acceptent un array `ks = np.arange(n + 1)`), dans une liste `gaps`, dans l'ordre de `N_LIST`.

Puis :
- a) l'écart pour $n = 10$ (4 décimales) ;
- b) l'écart pour $n = 1\,000$ (5 décimales) ;
- c) le rapport de l'écart pour $n = 100$ à l'écart pour $n = 1\,000$ (1 décimale) ;
- d) le plus petit $n$ de `N_LIST` pour lequel l'écart est inférieur à 0,01 (un entier).

Trace ensuite les écarts en fonction de $n$, en échelles logarithmiques sur les deux axes (`ax.loglog`).

Dans ta copie : quelle pente lis-tu sur ta courbe ? Que dit c) de la vitesse de convergence ? D'après b), à partir de quel $n$ l'approximation de Poisson te semble-t-elle suffisante pour calculer une probabilité à 2 décimales ?

In [ ]:
N_LIST = [5, 10, 30, 100, 1000]
LAM = 3

In [ ]:
gaps = ...           # the 5 gaps, in the order of N_LIST
gap_10 = ...         # a) a number
gap_1000 = ...       # b) a number
ratio_100_1000 = ... # c) a number
first_n = ...        # d) an integer of N_LIST

In [ ]:
wbm.check("M02b.23a", gap_10)
wbm.check("M02b.23b", gap_1000)
wbm.check("M02b.23c", ratio_100_1000)
wbm.check("M02b.23d", first_n)

**Le tracé** des écarts en fonction de $n$, en échelles logarithmiques (`ax.loglog`).

In [ ]:
# the plot of gaps against N_LIST, log scales on both axes

### Ex M02b.24 — Une loi jointe en carte de chaleur, ses marginales sur les bords 🎨 ★ ⏱️ 20 min
**Objectif :** représenter une loi jointe discrète par une carte de chaleur, avec ses deux lois marginales sur les bords, et mesurer l'écart à l'indépendance.  
**Prérequis :** fiche §5 ; R0 (matplotlib) · **Parcours :** C

On lance deux dés équilibrés ; $X$ est le plus petit des deux numéros, et $Y$ le plus grand (le même s'ils sont égaux). La cellule fournie construit leur loi jointe `JOINT_24` (un array 6 × 6 : la ligne $i$ correspond à $X = i + 1$, la colonne $j$ à $Y = j + 1$) et une fonction `joint_axes()` qui prépare une figure à trois axes.

Complète :
- `marginals(joint)`, qui renvoie le couple `(p_rows, p_cols)` des deux lois marginales : celle de la variable des lignes (somme de chaque ligne, `axis=1`) et celle de la variable des colonnes (`axis=0`) ;
- `draw_joint(ax, ax_top, ax_right, joint)`, qui dessine dans `ax` la carte de chaleur `ax.imshow(joint.T, origin="lower", cmap="Blues")` (la variable des lignes à l'horizontale), dans `ax_top` les barres de `p_rows` (`ax_top.bar`), et dans `ax_right` les barres horizontales de `p_cols` (`ax_right.barh`).

Puis calcule `max_gap_24`, le plus grand écart $\big\lvert P(X = x, Y = y) - P(X = x)\,P(Y = y) \big\rvert$ sur les 36 cases (`np.outer` fabrique le tableau des produits ; 4 décimales).

Dans ta copie : pourquoi la moitié de la carte est-elle vide ? Pourquoi la diagonale est-elle plus claire que les cases au-dessus ? $X$ et $Y$ sont-elles indépendantes ? Quelle case le montre le plus nettement, sans aucun calcul ?

In [ ]:
# The joint law of (the smaller, the larger) of two fair dice: rows = min (1 to 6), columns = max (1 to 6)
JOINT_24 = np.zeros((6, 6))
for a in range(1, 7):
    for b in range(1, 7):
        JOINT_24[min(a, b) - 1, max(a, b) - 1] += 1 / 36


def joint_axes():
    """A figure with three axes: the heat map, the margin of the rows above it, the margin of the columns on its
    right."""
    fig = plt.figure(figsize=(6.5, 5))
    grid = fig.add_gridspec(2, 2, width_ratios=(4, 1.3), height_ratios=(1.3, 4), wspace=0.08, hspace=0.08)
    ax = fig.add_subplot(grid[1, 0])
    return fig, ax, fig.add_subplot(grid[0, 0], sharex=ax), fig.add_subplot(grid[1, 1], sharey=ax)

In [ ]:
def marginals(joint):
    """The two margins (p_rows, p_cols) of a discrete joint law given as a 2D array."""
    raise NotImplementedError("joint.sum(axis=1) pour les lignes, joint.sum(axis=0) pour les colonnes")


def draw_joint(ax, ax_top, ax_right, joint):
    """Draw the heat map of joint in ax, the margin of the rows above (ax_top), the margin of the columns on the
    right (ax_right)."""
    raise NotImplementedError("ax.imshow(joint.T, origin=\"lower\"), ax_top.bar, ax_right.barh")


max_gap_24 = ...   # the largest |P(X = x, Y = y) − P(X = x) P(Y = y)| (4 decimals)

In [ ]:
with checked("M02b.24 figure"):
    fig, ax, ax_top, ax_right = joint_axes()
    try:
        created = drawn(draw_joint, ax, ax_top, ax_right, JOINT_24)
    except BaseException:
        plt.close(fig)                                     # no empty figure while the functions are not written yet
        raise
    if created:
        plt.close(fig)
        print("❌ Ex M02b.24 : draw_joint ouvre sa propre figure (plt.figure ou plt.subplots) : dessine dans les axes "
              "reçus.")
    else:
        ax.set(xlabel="X : le plus petit numéro", ylabel="Y : le plus grand numéro")
        ax.set_xticks(range(6), labels=range(1, 7))
        ax.set_yticks(range(6), labels=range(1, 7))
        plt.show()
wbm.check("M02b.24", max_gap_24)

**Vérifications automatiques** (la figure elle-même se juge à l'œil).

In [ ]:
with checked("M02b.24 vérifications"):
    problems = []
    test_joint = np.array([[0.1, 0.2, 0.0], [0.05, 0.15, 0.5]])      # 2 rows, 3 columns: a law that is not square
    margins = marginals(test_joint)
    if margins is None:
        problems.append("marginals ne renvoie rien (None) : il manque un return")
    else:
        p_rows, p_cols = (np.asarray(m, dtype=float) for m in margins)
        if p_rows.shape != (2,) or not np.allclose(p_rows, [0.3, 0.7]):
            problems.append("marginals : la loi des lignes somme chaque ligne (sur les colonnes), axis=1")
        if p_cols.shape != (3,) or not np.allclose(p_cols, [0.15, 0.35, 0.5]):
            problems.append("marginals : la loi des colonnes somme chaque colonne (sur les lignes), axis=0")
    fig, ax, ax_top, ax_right = joint_axes()
    try:
        created = drawn(draw_joint, ax, ax_top, ax_right, test_joint)
    finally:
        plt.close(fig)                                   # only the checks below are shown here
    if created:
        problems.append("draw_joint ouvre sa propre figure : dessine dans les trois axes reçus")
    else:
        images = ax.get_images()
        if not images or np.asarray(images[0].get_array()).shape != (3, 2) or not np.allclose(
                np.asarray(images[0].get_array()), test_joint.T):
            problems.append("draw_joint : la carte de chaleur ax.imshow(joint.T, origin=\"lower\"), les valeurs de "
                            "la première variable (les lignes de joint) à l'horizontale")
        top = sorted(round(p.get_height(), 6) for p in ax_top.patches)
        if top != [0.3, 0.7]:
            problems.append("draw_joint : au-dessus, les barres de la loi des lignes (ax_top.bar)")
        right = sorted(round(p.get_width(), 6) for p in ax_right.patches)
        if right != [0.15, 0.35, 0.5]:
            problems.append("draw_joint : à droite, les barres horizontales de la loi des colonnes (ax_right.barh)")
    if problems:
        print("❌ Ex M02b.24 : à revoir : " + " ; ".join(problems) + ".")
    else:
        print("✅ Ex M02b.24 : les marginales, la carte de chaleur et les deux barres des bords.")

## Partie C · Erreurs classiques, documentation et défi

Fiche §3, §6, §7 et §8. Trois confusions à trouver, une docstring testée, puis des normales fabriquées avec des uniformes.

### Ex M02b.25 — Chasse à l'erreur : une densité prise pour une probabilité, σ et σ² 🐛 ★ ⏱️ 15 min
**Objectif :** repérer trois confusions classiques sur les lois continues, puis calculer les bonnes valeurs.  
**Prérequis :** fiche §3, §4 et §6 · **Parcours :** C

Voici cinq affirmations, chacune avec le raisonnement de son auteur. Trois sont fausses. **Sans exécuter de code**, trouve-les.

- **(A)** « $X \sim \mathcal{N}(0 ;\ 0{,}1^2)$ a pour densité en 0 environ 3,99. C'est impossible, car une probabilité ne dépasse jamais 1 : la formule doit être fausse. »
- **(B)** « $X \sim \mathcal{N}(5, 4)$, donc $P(X \le 7) = \Phi\left(\frac{7 - 5}{4}\right) = \Phi(0{,}5) \approx 0{,}6915$. »
- **(C)** « $X \sim \mathcal{U}([0 ; 4])$ a pour variance $\frac{16}{12} = \frac{4}{3}$, donc pour écart-type $\approx 1{,}1547$. »
- **(D)** « $T \sim \mathcal{E}(2)$, donc $P(T = 1) = f(1) = 2e^{-2} \approx 0{,}2707$. »
- **(E)** « $X$ et $Y$ sont indépendantes, d'écarts-types 3 et 4, donc $\sigma(X + Y) = 5$. »

Questions :
- a) les lettres des affirmations fausses, dans l'ordre alphabétique (un texte) ;
- b) pour chacune des cinq, de (A) à (E), la bonne valeur de la grandeur dont parle son auteur : la densité en 0 pour (A), $P(X \le 7)$ pour (B), l'écart-type pour (C), $P(T = 1)$ pour (D), $\sigma(X + Y)$ pour (E) (une liste de 5 nombres, 4 décimales).

Dans ta copie : pour chaque affirmation fausse, quelle confusion a été faite ? Dans (A), la valeur 3,99 est-elle juste ?

In [ ]:
answer_M02b_25a = ...  # a) letters, as text
answer_M02b_25b = ...  # b) a list of 5 values (4 decimals)

In [ ]:
wbm.check("M02b.25a", answer_M02b_25a)
wbm.check("M02b.25b", answer_M02b_25b)

### Ex M02b.26 — Documenter une loi : docstring et exemples testés (`doctest`) 🛠️ ★ ⏱️ 15 min
**Objectif :** écrire une docstring au format NumPy dont les exemples sont exécutés comme des tests, réflexe professionnel pour une fonction de bibliothèque.  
**Prérequis :** fiche §2 ; les docstrings de `mymath` · **Parcours :** C

Les docstrings de `mymath` suivent le format NumPy (sections `Parameters`, `Returns`, `Raises`, `Examples`), et leurs exemples `>>>` sont des **tests** : le module `doctest` les exécute et compare le résultat affiché à celui qui est écrit. Une documentation testée ne ment pas.

Écris `geometric_pmf(k, p)`, la probabilité $P(X = k) = (1 - p)^{k-1}p$ de la loi géométrique (fiche §2), qui vaut `0.0` pour $k < 1$ et lève une `ValueError` si $p$ n'est pas dans $]0 ; 1]$, **avec sa docstring complète** : les quatre sections, et au moins trois exemples, dont une valeur ordinaire, le cas $k = 0$, et le cas d'erreur. Un exemple qui lève une erreur s'écrit ainsi :

```
>>> geometric_pmf(2, 1.5)
Traceback (most recent call last):
    ...
ValueError: p must be in ]0, 1]
```

(le message après `ValueError:` doit être exactement celui que lève ta fonction). Exécute ensuite tes exemples avec `doctest.run_docstring_examples(geometric_pmf, globals())` : rien ne s'affiche s'ils passent tous.

Dans ta copie : pourquoi un exemple de docstring qui n'est pas testé finit-il souvent par être faux ? Pourquoi écrire `round(..., 4)` dans un exemple qui affiche un flottant ?

In [ ]:
def geometric_pmf(k, p):
    """TODO: a one-line summary, then the sections Parameters, Returns, Raises and Examples (at least 3 examples)."""
    raise NotImplementedError("la formule (1 − p)^(k − 1) p, les cas k < 1 et p hors de ]0 ; 1], puis la docstring")

In [ ]:
with checked("M02b.26 vérifications"):
    import doctest

    problems = []
    doc = geometric_pmf.__doc__ or ""
    for word in ("Parameters", "Returns", "Raises", "Examples"):
        if word not in doc:
            problems.append(f"la docstring n'a pas de section « {word} » (le format NumPy des docstrings de mymath)")
    tests = doctest.DocTestFinder().find(geometric_pmf, "geometric_pmf", globs={"geometric_pmf": geometric_pmf})
    runner = doctest.DocTestRunner(optionflags=doctest.ELLIPSIS)
    for test in tests:
        runner.run(test, out=lambda text: None)
    attempted, failed = runner.tries, runner.failures
    if attempted < 3:
        problems.append(f"la docstring a {attempted} exemple(s) >>> : il en faut au moins 3 (une valeur ordinaire, "
                        "k = 0, et le cas d'erreur)")
    if failed:
        problems.append(f"{failed} exemple(s) de la docstring échoue(nt) : exécute "
                        "doctest.run_docstring_examples(geometric_pmf, globals()) pour les voir")
    if "ValueError" not in doc:
        problems.append("aucun exemple ne montre la ValueError (un exemple peut afficher Traceback ... ValueError)")
    cases = [(1, 0.25), (4, 0.25), (10, 0.6), (3, 1.0)]
    wrong = [(k, p) for k, p in cases if abs(geometric_pmf(k, p) - float(stats.geom(p).pmf(k))) > 1e-12]
    if wrong:
        problems.append(f"geometric_pmf se trompe, par exemple pour (k, p) = {wrong[0]} : (1 − p)^(k − 1) p")
    if geometric_pmf(0, 0.3) != 0.0 or geometric_pmf(-2, 0.3) != 0.0:
        problems.append("geometric_pmf(0, p) et geometric_pmf(-2, p) valent 0.0 : le premier succès a un rang au "
                        "moins égal à 1")
    if raised(geometric_pmf, 2, 0) != "ValueError" or raised(geometric_pmf, 2, 1.5) != "ValueError":
        problems.append("geometric_pmf lève une ValueError si p n'est pas dans ]0 ; 1]")
    if problems:
        print("❌ Ex M02b.26 : à revoir : " + " ; ".join(problems) + ".")
    else:
        print(f"✅ Ex M02b.26 : une docstring complète, {attempted} exemples qui passent, et une fonction juste.")

### Ex M02b.27 — Défi : tirer des valeurs normales avec seulement des tirages uniformes 🏆 ★★★ ⏱️ 30 min
**Objectif :** fabriquer des tirages de la loi normale centrée réduite à partir de nombres uniformes, assez fidèles pour passer un test statistique et reproduire les queues de la loi.  
**Prérequis :** fiche §7 et §8 ; Ex M02b.12 et M02b.20 · **Parcours :** C

Écris `normal_draws(rng, n)`, qui renvoie un array de $n$ tirages de $\mathcal{N}(0, 1)$ en n'utilisant **que** `rng.random(size)` : le générateur `UniformOnly` de la cellule fournie ne sait rien faire d'autre, et la vérification refuse `normal`, `scipy`, `ppf`, `erfinv`, `NormalDist` et un générateur à toi. **Objectifs**, vérifiés sur un million de valeurs :
1. le test de Kolmogorov-Smirnov (`stats.kstest`, qui compare la fonction de répartition des tirages à $\Phi$) ne rejette pas la loi normale : $p > 0{,}01$ sur 200 000 valeurs ;
2. les **queues** sont justes : la part des valeurs au-delà de 4 en valeur absolue est entre $3 \times 10^{-5}$ et $10^{-4}$ (la loi normale donne $2(1 - \Phi(4)) \approx 6{,}3 \times 10^{-5}$) ;
3. moins de 3 secondes pour un million de valeurs.

Pistes, de la plus simple à la plus élégante (`04_indices.md` les détaille) :
- additionner 12 uniformes et retrancher 6 (le théorème central limite) : tente-la, et regarde quel objectif elle manque, et pourquoi ;
- inverser $\Phi$ par dichotomie (M02a.18) : juste, mais lente, car `math.erf` ne traite qu'un nombre à la fois (essaie-la sur 1 000 valeurs) ;
- une transformation de **deux** uniformes en **deux** normales indépendantes, qui vient de l'intégrale de Gauss en coordonnées polaires (M02b.12, question 6 : la loi jointe de deux normales indépendantes ne dépend que de la distance au centre).

Dans ta copie : quelle méthode as-tu retenue, et pourquoi la somme de 12 uniformes échoue-t-elle sur les queues ? Pour la méthode polaire, quelle est la loi de $R^2 = X^2 + Y^2$ ?

In [ ]:
class UniformOnly:
    """A generator that can only draw uniform numbers in [0, 1): random(size)."""

    def __init__(self, seed):
        self._rng = np.random.default_rng(seed)

    def random(self, size=None):
        return self._rng.random(size)


FORBIDDEN_27 = ("normal", "standard_normal", "scipy", "ppf", "erfinv", "NormalDist", "default_rng", "Generator")

In [ ]:
def normal_draws(rng, n):
    """Return n draws of N(0, 1) as an array of shape (n,), using only rng.random(size)."""
    raise NotImplementedError("seulement rng.random : une somme d'uniformes, une dichotomie, ou deux uniformes en polaire")

In [ ]:
with checked("M02b.27"):
    import ast
    import inspect
    import time

    try:
        source = inspect.getsource(normal_draws)
    except (OSError, TypeError):
        source = ""
    names = {node.attr for node in ast.walk(ast.parse(source)) if isinstance(node, ast.Attribute)} | {
        node.id for node in ast.walk(ast.parse(source)) if isinstance(node, ast.Name)} if source else set()
    used = sorted(names & set(FORBIDDEN_27)) + sorted(word for word in ("scipy", "statistics") if word in source)
    start = time.perf_counter()
    big = np.asarray(normal_draws(UniformOnly(2026), 1_000_000), dtype=float)
    seconds = time.perf_counter() - start
    small = np.asarray(normal_draws(UniformOnly(7), 200_000), dtype=float)
    p_value = stats.kstest(small, "norm").pvalue if small.shape == (200_000,) else 0.0
    tail = float(np.mean(np.abs(big) > 4)) if big.shape == (1_000_000,) else 0.0
    print(f"forme : {big.shape} · test de Kolmogorov-Smirnov sur 200 000 valeurs : p = {p_value:.3f} · "
          f"part des |z| > 4 sur un million : {tail:.1e} (loi normale : 6,3e-05) · temps : {seconds:.2f} s")
    reasons = []
    if used:
        reasons.append(f"des outils interdits ({', '.join(sorted(set(used)))}) : seulement rng.random")
    if big.shape != (1_000_000,):
        reasons.append("un array de forme (n,)")
    if p_value <= 0.01:
        reasons.append("le test de Kolmogorov-Smirnov rejette la loi normale (p ≤ 0,01)")
    if not 3e-5 <= tail <= 1e-4:
        reasons.append("les queues ne sont pas celles d'une loi normale (valeurs au-delà de 4 écarts-types)")
    if seconds > 3:
        reasons.append("plus de 3 secondes pour un million de valeurs : vectorise avec NumPy")
    for reason in reasons:
        print(f"❌ {reason}")
    wbm.check("M02b.27", not reasons, computed=True)

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Reconnais-tu la loi d'une situation, et calcules-tu ses probabilités, son espérance et sa variance ?
2. Lis-tu une loi jointe, ses marginales, et sais-tu décider de l'indépendance ?
3. Sais-tu calculer la variance d'une somme ou d'une moyenne, et simuler une loi par inversion ?

**La suite** : M02c, les échantillons et l'estimation (loi des grands nombres, théorème central limite, intervalles de confiance, covariance), puis le chapitre 2 de workbookIA.